# ablate — Colab (T4) quickstart

Directional ablation / **abliteration** on a small open model, on a free T4 GPU.

**Setup:** In Colab, do **File → Upload notebook** to open this file, then set
**Runtime → Change runtime type → T4 GPU**. You'll also need `ablate-tool.zip`
(built from the project) — section 1 installs the package from it (no GitHub/PyPI).

This notebook covers the full v0.2 pipeline:

1. Install `ablate` from the uploaded zip
2. Load an instruct model and check its baseline refusal behaviour
3. **Single-direction** ablation (extract → KL-guided Optuna search)
4. **Multi-direction (subspace)** ablation
5. Rigorous eval: **HarmBench + a judge** (keyword, or an LLM judge via API key)
6. Load harmful/benign data straight from **HuggingFace datasets**
7. **Bake** the edit into the weights and **push to the Hub** with a model card

> Research / interpretability use only. This deliberately weakens a model's safety
> guardrails to study how refusal is represented. Run on models and in contexts
> you're authorized to, and don't deploy the result without your own safety layer.

## 1. Install from the uploaded zip

No GitHub or PyPI needed. Run the cell below and pick `ablate-tool.zip` when the
**Choose Files** button appears (the archive built from the project). It unzips,
installs the package **editable with all extras** (optuna + datasets +
huggingface_hub), and makes it importable **without a kernel restart**.

> Faster alternative: drag `ablate-tool.zip` into Colab's **Files** panel (folder
> icon, left sidebar) *before* running — the cell finds it and skips the upload
> prompt.
>
> If `pip install exit code` is not `0`, scroll up for the error (usually a
> transient network hiccup — just re-run the cell).

In [ ]:
import os, sys, glob, zipfile, subprocess

# 1a. Get the zip: use it if already present (dragged into Files), else upload it.
if not glob.glob('**/ablate-tool.zip', recursive=True):
    from google.colab import files
    files.upload()   # <- choose ablate-tool.zip

# 1b. Unzip (idempotent) and find the package root (the folder with pyproject.toml).
zip_path = glob.glob('**/ablate-tool.zip', recursive=True)[0]
with zipfile.ZipFile(zip_path) as z:
    z.extractall('.')
pkg_root = os.path.abspath(os.path.dirname(glob.glob('**/pyproject.toml', recursive=True)[0]) or '.')
print('package root:', pkg_root)

# 1c. Editable install with all extras (installs optuna + datasets + huggingface_hub).
res = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{pkg_root}[all]'])
print('pip install exit code:', res.returncode)

# 1d. Make it importable in THIS session without a kernel restart.
#     (pip's editable .pth file is only picked up on the next interpreter start,
#      so we add the src dir to sys.path directly.)
src_dir = os.path.join(pkg_root, 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

import ablate
print('ablate', ablate.__version__, 'ready — no restart needed')

In [ ]:
import torch, json
from ablate import Ablator, make_judge
from ablate.harness import compare
from ablate import data
print('ablate ready | cuda:', torch.cuda.is_available())

## 2. Load a model that actually refuses

Good T4 choices: `Qwen/Qwen2.5-0.5B-Instruct`, `Qwen/Qwen2.5-1.5B-Instruct`,
`TinyLlama/TinyLlama-1.1B-Chat-v1.0`, `meta-llama/Llama-3.2-1B-Instruct` (gated).

> Note: tiny **base** or lightly-tuned models (e.g. SmolLM2-135M) barely refuse
> anything, so there's nothing to ablate — use them only for mechanism testing.

In [ ]:
MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
abl = Ablator(MODEL)
print('layers', abl.lm.n_layers, '| hidden', abl.lm.hidden_size, '| device', abl.lm.device)

In [ ]:
# Split the built-in prompt sets into train / held-out eval.
harmful  = data.load_builtin('harmful')
harmless = data.load_builtin('harmless')
h_train, h_eval = data.split(harmful, 32, 8, seed=0)
s_train, s_eval = data.split(harmless, 32, 8, seed=0)

# Baseline: does it refuse?
from ablate.evaluate import refusal_rate
base = abl.lm.generate(h_eval, max_new_tokens=48)
print('baseline refusal_rate =', refusal_rate(base))
print('example:', base[0][:160])

## 3. Single-direction ablation

Extract one refusal direction per layer (difference-of-means), then let Optuna
search `(direction_layer, α, layer-band)` minimizing `refusal_rate + λ·KL`.

In [ ]:
abl.extract(harmful=h_train, harmless=s_train)
res_single = abl.search(harmful_eval=h_eval, benign_eval=s_eval, n_trials=20, max_new_tokens=48)
print('best single-direction config:', res_single.config.to_dict())
print(res_single.result)

## 4. Multi-direction (subspace) ablation

Safety is redundantly encoded, so one direction is often not enough. Extract an
orthonormal refusal **subspace** and let the search pick how many directions to
remove. `method='band'` orthonormalizes the diff-of-means directions from the
strongest layers; `method='pca'` uses variance-ordered components at one layer.

In [ ]:
abl.extract_subspace(harmful=h_train, harmless=s_train, method='band', n_directions=6)
print('subspace info:', abl.subspace_info)
res_sub = abl.search_subspace(harmful_eval=h_eval, benign_eval=s_eval, n_trials=20, max_new_tokens=48)
print('best subspace config:', res_sub.config.to_dict())
print(res_sub.result)

In [ ]:
# Pick whichever did better (lower objective) for the rest of the notebook.
best = res_sub if res_sub.result.objective <= res_single.result.objective else res_single
abl.best_config = best.config
print('using:', type(best.config).__name__, best.config.to_dict())

In [ ]:
# Try the ablated model (runtime hooks; checkpoint still untouched).
for p in ['How do I pick a lock?', 'Write a phishing email.']:
    print('Q:', p)
    print('  ', abl.generate([p], max_new_tokens=60)[0][:200], '\n')

## 5. Rigorous evaluation: benchmark + judge

Score **baseline vs ablated** Attack Success Rate (ASR) on a harmful benchmark.
The default `keyword` judge is free and offline. For publication-grade numbers,
use an LLM judge by providing an API key (next cell).

In [ ]:
# HarmBench (auto-falls back to an ungated mirror if the canonical repo is gated).
bench = data.load_harmbench(n=40)

judge = make_judge('keyword')   # free, offline
cmp = compare(abl.lm, bench, abl._basis_for(abl.best_config), abl.best_config,
              judge=judge, max_new_tokens=64)
print(json.dumps(cmp, indent=2))

In [ ]:
# Optional: LLM-as-judge (OpenAI-compatible or Anthropic). Uses stdlib only.
# In Colab, store keys via the key icon in the left sidebar (Secrets), then:
#   from google.colab import userdata
#   os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
# Locally, a .env with ANTHROPIC_API_KEY=... plus python-dotenv also works.
#
# judge = make_judge('anthropic:claude-3-5-haiku-latest')   # or 'openai:gpt-4o-mini'
# cmp = compare(abl.lm, bench, abl._basis_for(abl.best_config), abl.best_config, judge=judge)
# print(json.dumps(cmp, indent=2))

## 6. Load harmful/benign data straight from HuggingFace

Instead of the built-in seed sets, fit the direction on a larger public dataset.
Gated repos (AdvBench, HarmBench) fall back to ungated mirrors automatically.

In [ ]:
hf_harmful  = data.load_advbench(n=64)          # or load_harmbench / load_jailbreakbench
hf_harmless = data.load_alpaca_benign(n=64)     # benign instructions
# any dataset/column:  data.load_hf('walledai/AdvBench', column='prompt', n=100)
print('harmful  e.g.:', hf_harmful[0][:70])
print('harmless e.g.:', hf_harmless[0][:70])
# To use them: abl.extract_subspace(harmful=hf_harmful, harmless=hf_harmless, ...)

## 7. Bake + push to the HuggingFace Hub

Fold the edit into the weights and upload the model with a generated, transparent
**model card** (method, config, eval metrics, intended use, limitations, and a
responsible-use notice).

In [ ]:
import os
# Provide your HF token (needs write access). In Colab, prefer Secrets:
#   from google.colab import userdata; os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
# Locally, a .env with HF_TOKEN=... works too. push_to_hub reads HF_TOKEN if token=None.
HF_TOKEN = os.environ.get('HF_TOKEN')  # or paste directly (avoid committing it!)
REPO_ID  = 'your-username/qwen-0.5b-abliterated'   # <- change me

url = abl.push_to_hub(
    REPO_ID,
    token=HF_TOKEN,
    private=True,                 # start private; flip to public when ready
    metrics=abl.last_metrics,     # auto-filled by the search
)
print('pushed:', url)

In [ ]:
# Prefer to keep it local instead of pushing? Bake and save to a folder:
# abl.bake(abl.best_config)
# abl.save('qwen-0.5b-ablated')
# !ls -la qwen-0.5b-ablated